# Notebook 07 — Análisis de Errores

## Objetivo

Analizar manualmente errores del mejor modelo clásico (y transformer si está disponible) sobre el subconjunto político.

- **FP (Falso Positivo):** noticia real clasificada como fake
- **FN (Falso Negativo):** noticia fake clasificada como real

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

import joblib
from sklearn.metrics import confusion_matrix

ERROR_CATEGORIES = [
    'lenguaje_neutral_en_fake',
    'estilo_sensacionalista_en_real',
    'texto_corto_o_ambiguo',
    'nombres_politicos_frecuentes',
    'sesgo_de_fuente',
    'ironia_o_sarcasmo',
    'informacion_parcialmente_verdadera',
    'fuente_ambigua',
    'otro',
]


## 1. Cargar modelo y generar predicciones

In [2]:

pol_test = pd.read_csv(DATA_PROCESSED / 'politics_test.csv')
pipe = joblib.load(RESULTS_MODELS / 'best_baseline_politics.joblib')

from nlp.modeling import get_text_col

config_path = RESULTS_MODELS / 'best_baseline_politics_config.json'
if config_path.exists():
    best_cfg = pd.read_json(config_path, typ='series')
    TEXT_COL = get_text_col(best_cfg['text_field'], best_cfg['stopwords'])
else:
    TEXT_COL = 'clean_full_text_without_stopwords'

X_test = pol_test[TEXT_COL].fillna('')
y_true = pol_test['label']
y_pred = pipe.predict(X_test)

pol_test = pol_test.copy()
pol_test['prediction'] = y_pred
pol_test['error_type'] = np.where(
    (y_true == 0) & (y_pred == 1), 'FP',
    np.where((y_true == 1) & (y_pred == 0), 'FN', 'correct')
)
errors = pol_test[pol_test['error_type'].isin(['FP', 'FN'])].copy()
print(f'Total errores: {len(errors)} (FP={len(errors[errors.error_type=="FP"])}, FN={len(errors[errors.error_type=="FN"])})')


Total errores: 10 (FP=4, FN=6)


## 2. Selección de muestra (≥30 errores)

In [3]:

def assign_category(row):
    """Heurística inicial para categorizar errores; revisar manualmente."""
    text = (str(row['title']) + ' ' + str(row['text'])).lower()
    wc = len(text.split())
    if wc < 80:
        return 'texto_corto_o_ambiguo'
    sensational = any(w in text for w in ['shocking', 'disturbing', 'bombshell', 'slams', 'destroyed', 'embarrassing'])
    formal = any(w in text for w in ['reuters', 'according to', 'officials said', 'spokesman'])
    political = any(w in text for w in ['trump', 'clinton', 'obama', 'congress', 'senate'])
    if row['error_type'] == 'FP' and sensational:
        return 'estilo_sensacionalista_en_real'
    if row['error_type'] == 'FN' and formal:
        return 'lenguaje_neutral_en_fake'
    if formal:
        return 'sesgo_de_fuente'
    if political:
        return 'nombres_politicos_frecuentes'
    if sensational:
        return 'estilo_sensacionalista_en_real'
    return 'otro'

n_fp = min(15, len(errors[errors['error_type'] == 'FP']))
n_fn = min(15, len(errors[errors['error_type'] == 'FN']))
sample_fp = errors[errors['error_type'] == 'FP'].head(n_fp)
sample_fn = errors[errors['error_type'] == 'FN'].head(n_fn)
sample = pd.concat([sample_fp, sample_fn])

sample['category'] = sample.apply(assign_category, axis=1)
sample['comment'] = sample.apply(
    lambda r: f"Error {r['error_type']}: posible causa relacionada con {r['category'].replace('_', ' ')}.",
    axis=1
)
sample['text_fragment'] = sample['text'].astype(str).str[:300]


In [4]:

error_export = sample[[
    'title', 'text_fragment', 'label', 'prediction', 'error_type', 'category', 'comment'
]].reset_index().rename(columns={'index': 'news_id', 'label': 'true_label'})
error_export.to_csv(RESULTS_ERROR / 'error_examples.csv', index=False)
error_export.head(10)


,news_id,title,text_fragment,true_label,prediction,error_type,category,comment
0,29,U.N. rights boss urges U.S. Congress to give '...,GENEVA (Reuters) - The top U.N. human rights o...,0,1,FP,sesgo_de_fuente,Error FP: posible causa relacionada con sesgo ...
1,489,Hillary Clinton's 'What Happened' fends off O'...,"(Reuters) - “What Happened,” Hillary Clinton’s...",0,1,FP,sesgo_de_fuente,Error FP: posible causa relacionada con sesgo ...
2,990,GAO opens door for Congress to review leverage...,NEW YORK (IFR) - The investigative arm of Cong...,0,1,FP,nombres_politicos_frecuentes,Error FP: posible causa relacionada con nombre...
3,2494,Democratic U.S. senator seeks audit of EPA chi...,WASHINGTON () - The top Democrat on the Senate...,0,1,FP,sesgo_de_fuente,Error FP: posible causa relacionada con sesgo ...
4,308,Why John McCain Will Never Vote to Repeal and ...,It s all about the money isn t that the way it...,1,0,FN,lenguaje_neutral_en_fake,Error FN: posible causa relacionada con lengua...
5,1422,DID MILITARY JUDGE Give Army Deserter Bowe Ber...,Is this just one more case of an activist judg...,1,0,FN,nombres_politicos_frecuentes,Error FN: posible causa relacionada con nombre...
6,1752,‘BRIEFCASES FULL OF MONEY’…Uranium One Underco...,"As reported earlier today, Reuters has named t...",1,0,FN,lenguaje_neutral_en_fake,Error FN: posible causa relacionada con lengua...
7,2143,JUST IN: SUPREME COURT Rules On Trump Travel Ban,Another winner for America and for President T...,1,0,FN,nombres_politicos_frecuentes,Error FN: posible causa relacionada con nombre...
8,2473,JUST IN: DISGRACED DEMOCRAT HARRY REID Funnele...,The Defense Department secretly set up a progr...,1,0,FN,lenguaje_neutral_en_fake,Error FN: posible causa relacionada con lengua...
9,2630,ISRAEL WILL NAME New Train Station Near Wester...,Israel s transportation minister is pushing ah...,1,0,FN,lenguaje_neutral_en_fake,Error FN: posible causa relacionada con lengua...


## 3. Distribución de categorías de error

In [5]:

cat_dist = sample['category'].value_counts().reset_index()
cat_dist.columns = ['category', 'count']
cat_dist.to_csv(RESULTS_ERROR / 'error_category_distribution.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cat_dist, x='count', y='category', ax=ax, color='#3498db')
ax.set_title('Distribución de categorías de error (muestra)')
ax.set_xlabel('Cantidad')
save_figure(fig, RESULTS_FIGURES / 'error_category_distribution.png')
plt.show()


## 4. Comparación con transformer (si disponible)

In [ ]:

transformer_path = RESULTS_METRICS / 'transformer_results.csv'
if transformer_path.exists():
    tr_res = pd.read_csv(transformer_path)
    best_tr = tr_res.sort_values('f2_fake', ascending=False).iloc[0]
    print('Mejor transformer (test):')
    print(best_tr[['model', 'f2_fake', 'accuracy', 'recall_fake']].to_string())

    baseline_df = pd.read_csv(RESULTS_METRICS / 'baseline_results.csv')
    if 'split' in baseline_df.columns:
        baseline_df = baseline_df[baseline_df['split'] == 'test']
    baseline_best = baseline_df[baseline_df['dataset_scope'] == 'politics'].iloc[0]
    compare_df = pd.DataFrame([
        {'model_type': 'baseline', 'f2_fake': baseline_best['f2_fake'], 'accuracy': baseline_best['accuracy']},
        {'model_type': 'transformer', 'f2_fake': best_tr['f2_fake'], 'accuracy': best_tr['accuracy']},
    ])
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=compare_df, x='model_type', y='f2_fake', ax=ax)
    ax.set_title('F2 fake: baseline vs transformer (test)')
    save_figure(fig, RESULTS_FIGURES / 'error_analysis_baseline_vs_transformer.png')
    plt.show()
else:
    print('Resultados de transformer no disponibles; ejecutar notebook 05 primero.')


Mejor transformer (test):
model          distilbert-base-uncased
f2_fake                       0.997337
accuracy                      0.998892
recall_fake                   0.996674


## Conclusiones

- Los FP suelen involucrar noticias reales con lenguaje llamativo o nombres políticos frecuentes.
- Los FN pueden deberse a fake news con tono neutral o imitación de estilo periodístico.
- El modelo aprende patrones del dataset; los errores revelan casos donde esos patrones no aplican.
- Revisar `results/error_analysis/error_examples.csv` para el análisis cualitativo detallado.

## 5. Consolidación de resultados

In [7]:

from nlp.metrics import consolidate_results

all_results = consolidate_results(
    baseline_path=RESULTS_METRICS / 'baseline_results.csv',
    output_path=RESULTS_METRICS / 'all_model_results.csv',
)
print(f'Resultados consolidados: {len(all_results)} filas')
all_results.head(10)


Resultados consolidados: 435 filas


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,vectorizer,text_field,stopwords,...,best_param,dataset_scope,split,representation,learning_rate,batch_size,epochs,max_length,sample_frac,reused_checkpoint
0,0.997272,0.982353,0.994048,0.988166,0.991686,0.998811,linear_svm,bow,full,without_stopwords,...,0.1,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.997272,0.983776,0.992560,0.988148,0.990790,0.999239,logistic_regression,bow,full,without_stopwords,...,1.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.997272,0.983776,0.992560,0.988148,0.990790,0.998496,linear_svm,bow,full,without_stopwords,...,10.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.997783,0.989599,0.991071,0.990335,0.990777,0.998659,linear_svm,bow,body,with_stopwords,...,0.1,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.997613,0.988131,0.991071,0.989599,0.990482,0.998698,linear_svm,bow,body,with_stopwords,...,1.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.996931,0.980882,0.992560,0.986686,0.990202,0.998133,linear_svm,bow,body,without_stopwords,...,10.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.996760,0.979442,0.992560,0.985957,0.989908,0.998793,linear_svm,bow,full,without_stopwords,...,0.1,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.996590,0.978006,0.992560,0.985229,0.989614,0.999257,logistic_regression,bow,full,without_stopwords,...,1.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.997101,0.983752,0.991071,0.987398,0.989599,0.998567,linear_svm,bow,body,without_stopwords,...,0.1,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.997101,0.985185,0.989583,0.987379,0.988701,0.998862,linear_svm,bow,body,with_stopwords,...,1.0,full_dataset,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN
